In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import warnings

In [15]:
DATA_FILE = 'heart disease classification dataset.csv'
TARGET_COLUMN = 'target'
TEST_SIZE_RATIO = 0.30
RANDOM_SEED = 7777

In [16]:
try:
    # Load the dataset
    df = pd.read_csv(DATA_FILE)
    # The first column appears to be an index, drop it if it exists and is unnamed
    if df.columns[0] == df.columns[0]:
        df = df.iloc[:, 1:]
    
    print(f"Initial Shape: {df.shape}")
    print(f"Columns and Data Types:\n{df.info(verbose=False)}")
    print("-" * 50)
    
except FileNotFoundError:
    print(f"Error: The file '{DATA_FILE}' was not found.")
    exit()

Initial Shape: (303, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Columns: 14 entries, age to target
dtypes: float64(4), int64(8), object(2)
memory usage: 33.3+ KB
Columns and Data Types:
None
--------------------------------------------------


In [17]:
df = df.replace({'?': np.nan, '': np.nan, ' ': np.nan})
df = df.fillna(0)
print("Missing values filled with 0.")

Missing values filled with 0.


In [18]:
object_cols = df.select_dtypes(include='object').columns.tolist()

In [19]:
le = LabelEncoder()
df[TARGET_COLUMN] = le.fit_transform(df[TARGET_COLUMN])
print(f"Target variable '{TARGET_COLUMN}' encoded: {le.classes_} -> {le.transform(le.classes_)}")

Target variable 'target' encoded: ['no' 'yes'] -> [0 1]


In [20]:
if TARGET_COLUMN in object_cols:
    object_cols.remove(TARGET_COLUMN)

In [21]:
df = pd.get_dummies(df, columns=object_cols, drop_first=True)
print(f"Shape after One-Hot Encoding: {df.shape}")

Shape after One-Hot Encoding: (303, 14)


In [22]:
X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE_RATIO, random_state=RANDOM_SEED, stratify=y
)
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")

X_train shape: (212, 13)
X_test shape:  (91, 13)


In [24]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Features standardized.")

Features standardized.


In [26]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
y_pred = knn.predict(X_test_scaled)
print("KNN model trained and predictions made on the test set.")

KNN model trained and predictions made on the test set.


In [27]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

Accuracy:  0.8132
Precision: 0.8000
Recall:    0.8800
F1 Score:  0.8381


In [28]:
print(pd.DataFrame(conf_matrix, 
                   index=['Actual No Disease (0)', 'Actual Disease (1)'], 
                   columns=['Predicted No Disease (0)', 'Predicted Disease (1)']))

                       Predicted No Disease (0)  Predicted Disease (1)
Actual No Disease (0)                        30                     11
Actual Disease (1)                            6                     44
